# Playground for vetting and displaying the data

## Data Visualization with Matplotlib Tools


In [28]:
#@title "Data Visualization with Matplotlib Tools"
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta
# from zoneinfo import ZoneInfo
import pytz  # may need to migrate to ZoneInfo
### Global Structures and Configurations
# Timezone configuration OLD SCHOOL  But the Raspberry Pi OS may not support ZoneInfo
UTC = pytz.utc
EST = pytz.timezone('US/Eastern')
# Timezone configuration NEW SCHOOL
# UTC = ZoneInfo('UTC')
# EST = ZoneInfo('US/Eastern')
import matplotlib.pyplot as plt

import matplotlib.transforms
import matplotlib.dates as mdates


In [29]:
import logging
logging.basicConfig(level=logging.INFO)

## Develop the framework for the data retrieval and presentation

This is the structure that contains where the zones on the weather graphics are presented.

In [30]:
!scp pi@10.110.110.183:/home/pi/WeatherKiosk/resources/wave_data.csv ./tmp/
!scp pi@10.110.110.183:/home/pi/WeatherKiosk/resources/wind_data.csv ./tmp/
waveFile = "./tmp/wave_data.csv"
windFile = "./tmp/wind_data.csv"

wave_data.csv                                 100%   22KB 258.8KB/s   00:00    
wind_data.csv                                 100%   37KB 422.4KB/s   00:00    


In [40]:
windDF = pd.read_csv(windFile, index_col=0, parse_dates=True)
windDF.dropna(inplace=True) # wind data has glitches

In [ ]:
windDF.head(10)

Condition the data in preparation for plotting

In [41]:
# We need to average the components rather than the angles when resamping.
windDF['WdirSin'] = np.sin(np.radians(windDF['WindDir [°]']))
windDF['WdirCos'] = np.cos(np.radians(windDF['WindDir [°]']))

In [42]:
# Resample to 10 minute intervals, averaging the components
windAvgDF = windDF.select_dtypes('number').resample('1h').mean()

### Make Wind Graph

In [19]:
def makeWindGraph(windDF, whereFrom=""):
  if len(windDF) < 16:
    raise BaseException('Not enough points')
  
  # Work with data from the last 2 days
  cutoff_time = datetime.now(EST) - timedelta(days=2)
  windDF = windDF[windDF.index >= cutoff_time]

  # imageRef = pathToResources + 'tmp/' +  'windGraph.png' # fetch locally (way faster on a pi)
  imageRef = 'windGraph.png' # fetch locally (way faster on a pi)
  fig, ax = plt.subplots(figsize=(8, 4))

  tme = windDF.index
  wspd = windDF['WindSpeedAvg [kts]'] # windDF['WSPD']
  mxsp = windDF['WindSpeedGst [kts]'] # windDF['GST']

  # convert m/s to mph: 0.447, m/s to knot: 0.5144
  ax.plot(tme, wspd, 'bo-', alpha=0.8)
  ax.plot(tme, mxsp, 'ro-', alpha=0.8)

  # Plot direction arrows
  yloc = 3.0 * np.ones(windDF.shape[0])
  # we stored the direction components so the averages would be modulo 360 (or 2pi)
  #    The average between 10 and 350 should be 0 (or 360) NOT 180.
  # An arrow every other step
  (cosines, sines) = (windDF['WdirCos'], windDF['WdirSin'])
  ax.quiver(tme[::2], yloc[::2], sines[::2], cosines[::2],
            angles='uv', color='DodgerBlue', alpha=0.6, pivot='middle')
  # Set the axis labels
  # ax.set_xlabel('Date and Time', fontsize=10, fontstyle='italic', color='SlateGray')  #obvious don't need it.
  ax.set_ylabel('Wind Speed [knots]', fontsize=12, fontstyle='italic', color='SlateGray')

  #Fix the time axis
  ax.xaxis.set_major_locator(mdates.DayLocator(tz=EST))
  ax.xaxis.set_minor_locator(mdates.HourLocator(interval=4, tz=EST))

  ax.xaxis.set_major_formatter(mdates.DateFormatter('%a, %b %d', tz=EST))
  ax.xaxis.set_minor_formatter(mdates.DateFormatter('%H:%M', tz=EST))

  dx = 0.; dy = -10/72.
  offset = matplotlib.transforms.ScaledTranslation(dx, dy, fig.dpi_scale_trans)
  # Create offset transform by 5 points in x direction
  for label in ax.xaxis.get_majorticklabels():
      label.set(horizontalalignment='center', color='darkred', fontweight='bold')
      label.set_transform(label.get_transform() + offset)

  for label in ax.xaxis.get_minorticklabels():
      label.set(horizontalalignment='center', color='darkred')

  ax.grid(True, which='major', linewidth=2, axis='both', alpha=0.7)
  ax.grid(True, which='minor', linestyle='--', axis='both', alpha = 0.5)
  ax.set_ylim(bottom=0.0)

  # where did this come from?
  plt.text(0.99, 0.96, f"{whereFrom}",
        horizontalalignment='right', verticalalignment='center',
        transform=ax.transAxes, color='gray', alpha=0.6 )

  ##
  # Put a current conditions slug at the top
  tme = windDF.index[-1]
  wspd = np.round(2.23694 * windDF['WindSpeedAvg [kts]'].to_numpy()[-1],1)
  mxsp = np.round(2.23694 * windDF['WindSpeedGst [kts]'].to_numpy()[-1],1)
  # wspd = np.round(2.23694 * windDF['WSPD'].to_numpy()[-1][0],1)
  # mxsp = np.round(2.23694 * windDF['GST'].to_numpy()[-1][0],1)
  if mxsp != mxsp:
    mxsp = '-'
  temp = windDF['AirTemp [°F]'].to_numpy()[-1]
  wdir = windDF['WindSpeedAvg [m/s]'].to_numpy()[-1]
  # temp = windDF['AirTemp [°F]'].to_numpy()[-1][0]
  # wdir = windDF['WindSpeedAvg [m/s]'].to_numpy()[-1][0]
  old = datetime.now(tz=EST)-tme
  oldmin = np.int32(old.total_seconds()%60)
  oldhrs = np.int32(old.total_seconds()/3600)
  logging.debug(f"{tme}, {oldhrs}:{oldmin} old, {wspd} mph, {mxsp} mph, {wdir:4.0f}°T, {temp}°C")

  plt.text(0.99, 0.90, f"Last readings spd:{wspd}, max:{mxsp}, dir:{windDirection(wdir)}",
        horizontalalignment='right', verticalalignment='center',
        transform=ax.transAxes, color='blue', alpha=0.6 )
  if oldhrs > 1 or oldmin > 40:
    plt.text(0.99, 0.84, f"Warning {oldhrs}:{oldmin} old",
          horizontalalignment='right', verticalalignment='center',
          transform=ax.transAxes, color='darkred', alpha=0.6 )

#   fig.show()
  fig.savefig(imageRef, bbox_inches='tight', transparent=True)
  plt.close(fig)

# direction indexer
def windDirection(ang):
  labels = {
    'N': (-11.25, 11.25), 'NNE': (11.25, 33.75),   'NE': (33.75, 56.25),   'ENE': (56.25, 78.75),
    'E': (78.75, 101.25), 'ESE': (101.25, 123.75), 'SE': (123.75, 146.25), 'SSE': (146.25, 168.75),
    'S': (168.75, 191.25),'SSW': (191.25, 213.75), 'SW': (213.75, 236.25), 'WSW': (236.25, 258.75),
    'W': (258.75, 281.25),'WNW': (281.25, 303.75), 'NW': (303.75, 326.25), 'NNW': (326.25, 348.75),
    }
  for tag in labels.keys():
    if ang > labels[tag][0] and ang <= labels[tag][1]:
      return tag

# makeWindGraph(windDF, whereFrom="ExecRocks Buoy")

In [43]:
makeWindGraph(windAvgDF, whereFrom="XRocks Buoy")

In [44]:
import PIL.Image as Image
img = Image.open('windGraph.png')
img.show()

CinnamonDesktop-Message: 21:20:42.965: Ignoring thumbnailer with missing binary: 'xreader-thumbnailer'


### Waves

In [66]:
waveDF = pd.read_csv(waveFile, index_col=0, parse_dates=True)

 # We need to average the components rather than the angles when resamping.
waveDF['WdirSin'] = np.sin(np.radians(waveDF['WaveDir [°]']))
waveDF['WdirCos'] = np.cos(np.radians(waveDF['WaveDir [°]']))

# We need to trap some bad data. Wave periods greater than 5x the 24Avg are clearly misread.
needsAdjustment = (waveDF['WavPerAvg [s]'] / waveDF['WavePerAvgM24 [s]']) > 5.
waveDF.loc[needsAdjustment, 'WavPerAvg [s]']  = waveDF.loc[needsAdjustment, 'WavPerAvg [s]'] / 10.

# We need to trap some bad data. Wave periods greater than 5x the 24Avg are clearly misread.
needsAdjustment = (waveDF['WavPerDom [s]'] / waveDF['WavePerDomM24 [s]']) > 5.
waveDF.loc[needsAdjustment, 'WavPerDom [s]']  = waveDF.loc[needsAdjustment, 'WavPerDom [s]'] / 10.

# We need to trap some bad data. Wave heights greater than 3x the 24Avg are clearly misread.
needsAdjustment = (waveDF['WaveHgtMax [ft]'] / waveDF['WaveHgt24 [ft]']) > 3.
waveDF.loc[needsAdjustment, 'WaveHgtMax [ft]']  = waveDF.loc[needsAdjustment, 'WaveHgtMax [ft]'] / 100.


# waveDF.tail(10)
waveDF.head(10)

,WaveHgtSig [ft],WaveHgtMax [ft],WaveHgtSig [m],WaveHgtMax [m],WaveDir [°],WavPerAvg [s],WavPerDom [s],WaveHgt24 [ft],WaveDirM24 [°],WavePerAvgM24 [s],WavePerDomM24 [s],WaveTimeM24,WdirSin,WdirCos
TimeStamp,,,,,,,,,,,,,,
2026-01-27 19:56:00-05:00,0.54,0.96,0.17,0.29,232.0,2.4,2.0,3.22,310.0,2.6,2.6,2026-01-26 20:56:00-05:00,-0.788011,-0.615661
2026-01-27 20:16:00-05:00,0.50,0.88,0.15,0.27,251.0,2.0,2.0,3.22,310.0,2.6,2.6,2026-01-26 20:56:00-05:00,-0.945519,-0.325568
2026-01-27 20:36:00-05:00,0.55,0.97,0.17,0.30,239.0,2.0,2.0,3.22,310.0,2.6,2.6,2026-01-26 20:56:00-05:00,-0.857167,-0.515038
2026-01-27 20:56:00-05:00,0.58,1.03,0.18,0.31,245.0,2.0,2.0,2.93,226.0,2.5,2.7,2026-01-27 17:16:00-05:00,-0.906308,-0.422618
2026-01-27 21:16:00-05:00,0.67,1.18,0.20,0.36,252.0,2.0,2.0,2.93,226.0,2.5,2.7,2026-01-27 17:16:00-05:00,-0.951057,-0.309017
2026-01-27 21:36:00-05:00,0.67,1.18,0.20,0.36,261.0,2.0,2.2,2.93,226.0,2.5,2.7,2026-01-27 17:16:00-05:00,-0.987688,-0.156434
2026-01-27 21:56:00-05:00,0.97,1.72,0.30,0.52,261.0,2.4,2.2,2.93,226.0,2.5,2.7,2026-01-27 17:16:00-05:00,-0.987688,-0.156434
2026-01-27 22:16:00-05:00,0.91,1.60,0.28,0.49,242.0,2.4,2.2,2.93,226.0,2.5,2.7,2026-01-27 17:16:00-05:00,-0.882948,-0.469472
2026-01-27 22:36:00-05:00,0.86,1.51,0.26,0.46,283.0,2.4,2.2,2.93,226.0,2.5,2.7,2026-01-27 17:16:00-05:00,-0.974370,0.224951


### Make Wave Graph

In [ ]:
def makeWaveGraph(waveDF, whereFrom=""):
  if len(waveDF) < 16:
    raise BaseException('Not enough points')
  
  # Work with data from the last 2 days
  cutoff_time = datetime.now(EST) - timedelta(days=2)
  waveDF = waveDF[waveDF.index >= cutoff_time]

  # imageRef = pathToResources + 'tmp/' +  'windGraph.png' # fetch locally (way faster on a pi)
  imageRef = 'waveGraph.png' # fetch locally (way faster on a pi)
  fig, ax = plt.subplots(figsize=(8, 4))


  tme = waveDF.index
  wspd = waveDF['WavePerAvg [s]'] 
  mxsp = waveDF['WaveDom [s]'] # waveDF['GST']

  # convert m/s to mph: 0.447, m/s to knot: 0.5144
  ax.plot(tme, wspd, 'bo-', alpha=0.8)
  ax.plot(tme, mxsp, 'ro-', alpha=0.8)

  # Plot direction arrows
  yloc = 3.0 * np.ones(waveDF.shape[0])
  # we stored the direction components so the averages would be modulo 360 (or 2pi)
  #    The average between 10 and 350 should be 0 (or 360) NOT 180.
  # An arrow every other step
  (cosines, sines) = (waveDF['WdirCos'], waveDF['WdirSin'])
  ax.quiver(tme[::2], yloc[::2], sines[::2], cosines[::2],
            angles='uv', color='DodgerBlue', alpha=0.6, pivot='middle')
  # Set the axis labels
  # ax.set_xlabel('Date and Time', fontsize=10, fontstyle='italic', color='SlateGray')  #obvious don't need it.
  ax.set_ylabel('Wave Period [s]', fontsize=12, fontstyle='italic', color='SlateGray')

  #Fix the time axis
  ax.xaxis.set_major_locator(mdates.DayLocator(tz=EST))
  ax.xaxis.set_minor_locator(mdates.HourLocator(interval=4, tz=EST))

  ax.xaxis.set_major_formatter(mdates.DateFormatter('%a, %b %d', tz=EST))
  ax.xaxis.set_minor_formatter(mdates.DateFormatter('%H:%M', tz=EST))

  dx = 0.; dy = -10/72.
  offset = matplotlib.transforms.ScaledTranslation(dx, dy, fig.dpi_scale_trans)
  # Create offset transform by 5 points in x direction
  for label in ax.xaxis.get_majorticklabels():
      label.set(horizontalalignment='center', color='darkred', fontweight='bold')
      label.set_transform(label.get_transform() + offset)

  for label in ax.xaxis.get_minorticklabels():
      label.set(horizontalalignment='center', color='darkred')

  ax.grid(True, which='major', linewidth=2, axis='both', alpha=0.7)
  ax.grid(True, which='minor', linestyle='--', axis='both', alpha = 0.5)
  ax.set_ylim(bottom=0.0)

  # where did this come from?
  plt.text(0.99, 0.96, f"{whereFrom}",
        horizontalalignment='right', verticalalignment='center',
        transform=ax.transAxes, color='gray', alpha=0.6 )

  ##
  # Put a current conditions slug at the top
  tme = waveDF.index[-1]
  wspd = np.round(waveDF['WaveHeight [m]'].to_numpy()[-1],1)
  mxsp = np.round(waveDF['WavPerAvg [s]'].to_numpy()[-1],1)
  # wspd = np.round(2.23694 * waveDF['WSPD'].to_numpy()[-1][0],1)
  # mxsp = np.round(2.23694 * waveDF['GST'].to_numpy()[-1][0],1)
  if mxsp != mxsp:
    mxsp = '-'
  temp = waveDF['AirTemp [°F]'].to_numpy()[-1]
  wdir = waveDF['WindSpeedAvg [m/s]'].to_numpy()[-1]
  # temp = waveDF['AirTemp [°F]'].to_numpy()[-1][0]
  # wdir = waveDF['WindSpeedAvg [m/s]'].to_numpy()[-1][0]
  old = datetime.now(tz=EST)-tme
  oldmin = np.int32(old.total_seconds()%60)
  oldhrs = np.int32(old.total_seconds()/3600)
  logging.debug(f"{tme}, {oldhrs}:{oldmin} old, {wspd} mph, {mxsp} mph, {wdir:4.0f}°T, {temp}°C")

  plt.text(0.99, 0.90, f"Last readings spd:{wspd}, max:{mxsp}, dir:{windDirection(wdir)}",
        horizontalalignment='right', verticalalignment='center',
        transform=ax.transAxes, color='blue', alpha=0.6 )
  if oldhrs > 1 or oldmin > 40:
    plt.text(0.99, 0.84, f"Warning {oldhrs}:{oldmin} old",
          horizontalalignment='right', verticalalignment='center',
          transform=ax.transAxes, color='darkred', alpha=0.6 )

#   fig.show()
  fig.savefig(imageRef, bbox_inches='tight', transparent=True)
  plt.close(fig)

# direction indexer
def windDirection(ang):
  labels = {
    'N': (-11.25, 11.25), 'NNE': (11.25, 33.75),   'NE': (33.75, 56.25),   'ENE': (56.25, 78.75),
    'E': (78.75, 101.25), 'ESE': (101.25, 123.75), 'SE': (123.75, 146.25), 'SSE': (146.25, 168.75),
    'S': (168.75, 191.25),'SSW': (191.25, 213.75), 'SW': (213.75, 236.25), 'WSW': (236.25, 258.75),
    'W': (258.75, 281.25),'WNW': (281.25, 303.75), 'NW': (303.75, 326.25), 'NNW': (326.25, 348.75),
    }
  for tag in labels.keys():
    if ang > labels[tag][0] and ang <= labels[tag][1]:
      return tag

# makeWindGraph(waveDF, whereFrom="ExecRocks Buoy")